In [1]:
%pip install pandas datasets torch transformers peft scikit-learn evaluate numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 10.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.0/865.0 MB 8.8 MB/s eta 0:00:0000:0100:03m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 9.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 10.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 10.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 10.0 MB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 9.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 10.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
from datasets import load_from_disk, load_dataset
# Load the dataset
# Save the dataset locally
# dataset = load_dataset('fancyzhx/ag_news')
dataset = load_from_disk('ag_news')

dataset = dataset.rename_column('label','labels')


# Load the dataset from local
dataset['train'] = dataset['train'].shuffle(seed=42).select(range(500))
dataset['test'] = dataset['test'].shuffle(seed=42).select(range(100))

dataset['train']

Dataset({
    features: ['text', 'labels'],
    num_rows: 500
})

In [9]:
import torch
print(torch.cuda.is_available())  # Ensure CUDA is available
print(torch.cuda.current_device())  # Check the current device
print(torch.__version__)  # Check PyTorch version
print(torch.cuda.get_device_name(0)) 

True
0
2.7.0+cu126
NVIDIA GeForce GTX 1050 Ti


In [10]:
from transformers import AutoTokenizer, BertTokenizer


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
train_dataset = dataset['train']
test_dataset = dataset['train']
X = tokenizer(train_dataset['text'], padding=True, truncation=True, max_length=512)
X1 = tokenizer(test_dataset['text'], padding=True, truncation=True, max_length=512)
y = train_dataset['labels']
y1 = test_dataset['labels']

In [11]:
import torch
# Create torch dataset
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

In [12]:
train_dataset = Dataset(X, y)
test_dataset = Dataset(X1, y1)

In [13]:
from transformers import TrainingArguments


# Define the training arguments
training_args = TrainingArguments(
    output_dir="./results1",            # Where to save the model checkpoints
    eval_strategy='steps',
    # learning_rate=2e-5,                # Learning rate for training
    per_device_train_batch_size=8,    # Batch size for training
    per_device_eval_batch_size=8,     # Batch size for evaluation
    num_train_epochs=1,                # Number of epochs
    # weight_decay=0.01,                 # Weight decay for regularization
    logging_dir="./logs",              # Where to save logs
    logging_steps=10,                  # Log every 10 steps
    label_names=['labels'],
    # load_best_model_at_end=True,       # Load the best model at the end of training
    metric_for_best_model="accuracy"   # Recommended metric for classification tasks
)


In [14]:


from peft import get_peft_model, LoraConfig
from transformers import BertForSequenceClassification
# Load the BERT base uncased model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)
# lora_config = LoraConfig(
#     r=4,  # Low-rank size, you can experiment with this value
#     lora_alpha=32,  # Scaling factor, controlling the impact of LoRa
#     lora_dropout=0.1,  # Dropout rate for LoRa layers
# )
# # Apply LoRa configuration using get_peft_model
# model = get_peft_model(base_model, lora_config)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
from transformers import Trainer
from sklearn.metrics import accuracy_score
from evaluate import load
import numpy as np

accuracy = load("accuracy")

# Load metric
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    # return {'accuracy': acc, 'eval_accuracy': acc}
    return accuracy.compute(predictions=predictions, references=labels)
# Prepare the trainer
trainer = Trainer(
    model=model,   
    args=training_args,                    # Training arguments defined earlier
    train_dataset=train_dataset,   # Training dataset
    eval_dataset=test_dataset,     # Evaluation dataset
    tokenizer=tokenizer,   
    compute_metrics=compute_metrics  # Accuracy metric

)

/tmp/ipykernel_19060/2146187270.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [16]:
import os, torch
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

trainer.train()

Step,Training Loss,Validation Loss,Accuracy
10,1.311700,1.205125,0.600000
20,1.164300,1.013092,0.718000
30,0.880500,0.681038,0.870000
40,0.648300,0.541016,0.912000
50,0.600700,0.435278,0.908000
60,0.551100,0.403585,0.920000


TrainOutput(global_step=63, training_loss=0.8317138647276258, metrics={'train_runtime': 391.2229, 'train_samples_per_second': 1.278, 'train_steps_per_second': 0.161, 'total_flos': 50105055780000.0, 'train_loss': 0.8317138647276258, 'epoch': 1.0})

In [17]:
trainer.evaluate()

{'eval_loss': 0.39889776706695557,
 'eval_accuracy': 0.918,
 'eval_runtime': 40.0172,
 'eval_samples_per_second': 12.495,
 'eval_steps_per_second': 1.574,
 'epoch': 1.0}

In [18]:
def predict_label(text: str, model, tokenizer):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_id = torch.argmax(logits, dim=-1).item()
    
    # print("Predicted Label:", predicted_class_id)
    return predicted_class_id
    

In [20]:

test = dataset['test'].shuffle(seed=12).select(range(40))
cnt_matched = 0
for i in test:
    ans = predict_label(i['text'], trainer.model, trainer.tokenizer)
    if ans == i['labels']:
        cnt_matched += 1
        print("Matched the label")
        print(i['text'])
        print("Dataset Label:" + str(i['labels']))
        print("Predicted Label: " + str(ans))

print(f"Matched Results: {cnt_matched} / 40")

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Matched the label
Hokies making statement about ACC title intentions The annual summer barbecues that Ralph Friedgen and Frank Beamer co-host at their lake homes in Georgia may be a little less cordial after the way Beamer #39;s Virginia Tech Hokies waxed Friedgen #39;s Maryland Terrapins 55-6 last night.
Dataset Label:1
Predicted Label: 1
Matched the label
Putin praises Ukraine #39;s leader Russian President Vladimir Putin has taken part in a live phone-in on Ukrainian TV, just days before the country #39;s presidential election.
Dataset Label:0
Predicted Label: 0
Matched the label
Thrashers Owner Fined The NHL fined one of the owners of the Thrashers \$250,000 on Tuesday for saying the league would use replacement players next year if a new collective bargaining agreement isn't reached.
Dataset Label:1
Predicted Label: 1
Matched the label
James Lawton: In mourning for big fights of Las Vegas En route to Las Vegas for the world heavyweight title fight between Vitali Klitschko and Brit

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Matched the label
Rebound in US consumer spending US consumer spending rebounded in July, a sign the economy may be emerging from an early summer decline. Consumer spending rose 0.8 last month, boosted by car and retail sales.
Dataset Label:2
Predicted Label: 2
Matched the label
ADV: Try Currency Trading Risk-Free 30 Days 24-hour commission-free trading, 100-to-1 leverage of your capital, and Dealbook Fx 2 - our free advanced trading software. Sign up for our free 30-day trial and receive one-on-one training.
Dataset Label:2
Predicted Label: 2
Matched the label
Israel Destroys Refugee Homes, Kills One GAZA CITY, Gaza Strip - A day after a mortar round killed an Israeli-American woman in a nearby settlement, the Israeli army charged into a Palestinian refugee camp Saturday, killing one person and tearing down 35 homes, witnesses and a U.N. aid official said...
Dataset Label:0
Predicted Label: 0
Matched the label
UN 'must ignore cloning ban call' The UK's Royal Society urges the UN to ig

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


...

Matched the label
China's inflation rate slows sharply but problems remain (AFP) AFP - China's inflation rate eased sharply in October as government efforts to cool the economy began to really bite, with food prices, one of the main culprits, showing some signs of slowing, official data showed.
Dataset Label:0
Predicted Label: 0
Matched the label
Oil Prices Sink to a Four-Month Low  LONDON (Reuters) - Oil prices sank to a four-month low  below \$41 for U.S. crude on Wednesday after leading OPEC  producer Saudi Arabia questioned the need for the cartel to  curb supplies.
Dataset Label:2
Predicted Label: 2
Matched the label
Bush won #39;t take Iran #39;s word for it CRAWFORD, Texas As President Bush sees it,  quot;the only good deal is one that #39;s verifiable. quot;. He #39;s applauding the efforts of some European countries to get Iran to honor its commitment to refrain from developing nuclear weapons.
Dataset Label:0
Predicted Label: 0
Matched the label
Bellhorn Makes Big Noise for R

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Matched the label
Headshake to the SETI Headfake Did the famous screensaver, SETIhome, uncover the first strong evidence for an extraterrestrial signal? The SETI Institute's Seth Shostak discusses how hyperbole can misrepresent the last addition to a list of stellar candidates.
Dataset Label:3
Predicted Label: 3
Matched the label
A PC in the toaster? How mod! photos There's also room in the humidor and the Darth Vader helmet. Take a gander at some strange and wonderful creations.
Dataset Label:3
Predicted Label: 3
Matched the label
Oil price rise adds to airlines woes as losses continue The global airline industry is forecast to have made net losses of \$4.8bn this year, as the rise in the oil price has overwhelmed efforts by carriers to cut costs.
Dataset Label:2
Predicted Label: 2
Matched the label
NASA Cancels Hypersonic Flight NASA scrubbed its mission Monday to launch a pilotless plane that is capable of flying at 10 times the speed of sound. The launch of the X-43A was canceled d

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Matched the label
Gold Fields loses high court bid to halt Harmony takeover South African mining giant Gold Fields lost a high court bid to halt a hostile takeover by rival Harmony Gold, which is seeking to create the world #39;s biggest gold producer, a court official said.
Dataset Label:2
Predicted Label: 2
Matched the label
SCHOOL #39;S OUT FOR STERNE A first win on the European Tour - any tour, in fact - is a notable feat in any golfer #39;s career. But the one by South African Richard Sterne in the Madrid Open yesterday deserves special mention.
Dataset Label:1
Predicted Label: 1
Matched the label
Ex-Philly Eagles Coach Nick Skorich Dies (AP) AP - Nick Skorich, head coach of the Philadelphia Eagles from 1961-63 and the offensive line coach on the 1960 championship team, has died at the age of 83.
Dataset Label:1
Predicted Label: 1
Matched the label
Roundup: Pleasantly Perfect takes Pacific Classic Favored Pleasantly Perfect took charge down the stretch to win by a length in the 14